# Practical 12 - Geospatial Queries

This notebook works with a deterministic set of synthetic places around central
London and a separate set of hand-authored district polygons. It covers
`geo_point` mapping, radius and bounding-box searches, distance sorting,
geo-distance aggregations, and `geo_shape`/polygon queries.

**Tasks**

1. Create an index with `geo_point` mapping.
2. Insert records containing latitude and longitude.
3. Search within a 5 km radius.
4. Search within a geographical bounding box.
5. Sort results by distance.
6. Implement geo-distance aggregations.
7. Explore `geo_shape` and polygon queries.

All coordinates are synthetic. The place names and businesses are generated;
they are not real locations.

## Setup

The notebook must be executed from the `Practical_12` directory. It creates
`p12-locations` and `p12-districts`, leaving the indices from earlier
practicals untouched.

In [1]:
import json
import math
import statistics
import time
from pathlib import Path

from elasticsearch import Elasticsearch, helpers

from generate_data import (
    CENTER,
    DISTRICT_FIELDS,
    MAX_RADIUS_KM,
    PLACE_FIELDS,
    build_districts,
    build_places,
)

HERE = Path.cwd()
assert HERE.name == "Practical_12", f"Run this notebook from Practical_12, not {HERE}"

ARTIFACTS = HERE / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

ES_URL = "http://127.0.0.1:9200"
TOTAL = 10_000
LOCATIONS = "p12-locations"
DISTRICTS = "p12-districts"
REFERENCE = {"lat": CENTER["lat"], "lon": CENTER["lon"]}
RADIUS_KM = 5.0
BBOX = {
    "top_left": {"lat": 51.54, "lon": -0.18},
    "bottom_right": {"lat": 51.48, "lon": -0.07},
}
DISTANCE_RANGES = [
    {"to": 1},
    {"from": 1, "to": 2},
    {"from": 2, "to": 5},
    {"from": 5, "to": 10},
    {"from": 10, "to": 20},
    {"from": 20},
]
SORT = [
    {
        "_geo_distance": {
            "location": REFERENCE,
            "order": "asc",
            "unit": "km",
            "mode": "min",
        }
    }
]

MAPPING = json.loads((HERE / "mapping.json").read_text(encoding="utf-8"))
LOCATION_MAPPING = MAPPING["locations"]
DISTRICT_MAPPING = MAPPING["districts"]

es = Elasticsearch(ES_URL, request_timeout=120)
info = es.info()
print("Connected to Elasticsearch", info["version"]["number"])
print("cluster:", info["cluster_name"], "| node:", info["name"])
print("reference point:", REFERENCE)
print("dataset size:", f"{TOTAL:,}", "| maximum generated radius:", MAX_RADIUS_KM, "km")


def plain(value):
    if hasattr(value, "items"):
        return {key: plain(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(item) for item in value]
    return value


def delete_index(name):
    if es.indices.exists(index=name):
        es.indices.delete(index=name)


def bulk_load(index, records, id_field, chunk_size=2000):
    started = time.perf_counter()
    loaded, errors = helpers.bulk(
        es,
        (
            {"_index": index, "_id": record[id_field], "_source": record}
            for record in records
        ),
        chunk_size=chunk_size,
        raise_on_error=False,
    )
    es.indices.refresh(index=index)
    seconds = time.perf_counter() - started
    return {
        "documents": int(loaded),
        "errors": len(errors) if isinstance(errors, list) else int(errors),
        "seconds": round(seconds, 3),
        "docs_per_second": round(loaded / seconds, 1) if seconds else 0.0,
    }


def index_metrics(index):
    stats = plain(es.indices.stats(index=index))["indices"][index]["primaries"]
    return {
        "documents": int(stats["docs"]["count"]),
        "primary_store_bytes": int(stats["store"]["size_in_bytes"]),
        "segments": int(stats["segments"]["count"]),
    }


def haversine_km(origin, destination):
    radius = 6371.0088
    lat1 = math.radians(origin["lat"])
    lat2 = math.radians(destination["lat"])
    delta_lat = math.radians(destination["lat"] - origin["lat"])
    delta_lon = math.radians(destination["lon"] - origin["lon"])
    a = (
        math.sin(delta_lat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(delta_lon / 2) ** 2
    )
    return 2 * radius * math.asin(math.sqrt(a))


def hit_summary(hit):
    source = hit["_source"]
    result = {
        "place_id": source["place_id"],
        "name": source["name"],
        "category": source["category"],
        "latitude": source["latitude"],
        "longitude": source["longitude"],
        "location": source["location"],
        "rating": source["rating"],
    }
    if "sort" in hit:
        result["distance_km"] = round(float(hit["sort"][0]), 3)
    return result

Connected to Elasticsearch 9.2.0
cluster: my-elasticsearch-cluster | node: my-node
reference point: {'lat': 51.5074, 'lon': -0.1278}
dataset size: 10,000 | maximum generated radius: 20.0 km


## Task 1 - Create an index with `geo_point` mapping

The main index uses an explicit strict mapping, one shard, and zero replicas.
Both numeric latitude/longitude fields and a dedicated `location` object are
stored. Queries use `location`; the numeric fields make the source easy to
inspect and validate.

In [2]:
delete_index(LOCATIONS)
es.indices.create(index=LOCATIONS, **LOCATION_MAPPING)

locations_mapping = plain(es.indices.get_mapping(index=LOCATIONS))[LOCATIONS]["mappings"]
locations_settings = plain(es.indices.get_settings(index=LOCATIONS))[LOCATIONS]["settings"]["index"]
location_type = locations_mapping["properties"]["location"]["type"]

print("index:", LOCATIONS)
print("location mapping:", json.dumps(locations_mapping["properties"]["location"], indent=2))
print("shards:", locations_settings["number_of_shards"], "| replicas:", locations_settings["number_of_replicas"])
print("dynamic:", locations_mapping.get("dynamic"))

assert location_type == "geo_point"
assert locations_mapping["properties"]["latitude"]["type"] == "double"
assert locations_mapping["properties"]["longitude"]["type"] == "double"
assert locations_mapping["dynamic"] == "strict"
assert locations_settings["number_of_shards"] == "1"
assert locations_settings["number_of_replicas"] == "0"
print("geo_point mapping verified.")

index: p12-locations
location mapping: {
  "type": "geo_point"
}
shards: 1 | replicas: 0
dynamic: strict
geo_point mapping verified.


## Task 2 - Insert records containing latitude and longitude

Ten thousand deterministic places are generated inside a 20 km disk around
central London. The load runs with refresh disabled, then refresh is restored
to `1s`.

In [3]:
load_info = bulk_load(LOCATIONS, build_places(count=TOTAL), "place_id", chunk_size=2000)
es.indices.put_settings(index=LOCATIONS, settings={"refresh_interval": "1s"})
es.indices.refresh(index=LOCATIONS)
indexed_count = int(es.count(index=LOCATIONS, query={"match_all": {}})["count"])
first_document = plain(es.get(index=LOCATIONS, id="PLC-000001"))["_source"]
location_metrics = index_metrics(LOCATIONS)

print("load:", load_info)
print("count:", f"{indexed_count:,}")
print("first document:")
print(json.dumps(first_document, indent=2))

assert load_info["errors"] == 0
assert indexed_count == TOTAL
assert first_document["location"] == {
    "lat": first_document["latitude"],
    "lon": first_document["longitude"],
}
print("load and document shape verified.")

load: {'documents': 10000, 'errors': 0, 'seconds': 3.049, 'docs_per_second': 3279.6}
count: 10,000
first document:
{
  "place_id": "PLC-000001",
  "name": "Islington Bank 00001",
  "category": "bank",
  "latitude": 51.529884,
  "longitude": 0.100176,
  "location": {
    "lat": 51.529884,
    "lon": 0.100176
  },
  "rating": 3.4,
  "capacity": 382,
  "open_now": true
}
load and document shape verified.


## Task 3 - Search within a 5 km radius

The `geo_distance` query is centered on the reference point. The result is also
checked independently with a haversine distance calculation.

In [4]:
RADIUS_QUERY = {
    "geo_distance": {
        "distance": f"{RADIUS_KM:g}km",
        "location": REFERENCE,
    }
}
radius_started = time.perf_counter()
radius_response = es.search(
    index=LOCATIONS,
    query=RADIUS_QUERY,
    sort=SORT,
    size=2000,
    track_total_hits=True,
)
radius_seconds = time.perf_counter() - radius_started
radius_total = int(radius_response["hits"]["total"]["value"])
radius_hits = radius_response["hits"]["hits"]
radius_distances = [
    haversine_km(REFERENCE, hit["_source"]["location"]) for hit in radius_hits
]

print("radius:", RADIUS_KM, "km")
print("matches:", f"{radius_total:,}")
print("returned hits:", len(radius_hits))
print("maximum haversine distance:", round(max(radius_distances), 4), "km")
print("nearest five:")
print(json.dumps([hit_summary(hit) for hit in radius_hits[:5]], indent=2))

assert radius_total > 0
assert radius_total < TOTAL
assert radius_total == len(radius_hits)
assert max(radius_distances) <= RADIUS_KM + 0.01
print("all returned places are within the 5 km radius.")

radius: 5.0 km
matches: 660
returned hits: 660
maximum haversine distance: 4.9907 km
nearest five:
[
  {
    "place_id": "PLC-007804",
    "name": "Thames Bank 07804",
    "category": "bank",
    "latitude": 51.505577,
    "longitude": -0.129103,
    "location": {
      "lat": 51.505577,
      "lon": -0.129103
    },
    "rating": 4.9,
    "distance_km": 0.222
  },
  {
    "place_id": "PLC-000562",
    "name": "Hampstead Bank 00562",
    "category": "bank",
    "latitude": 51.508665,
    "longitude": -0.124636,
    "location": {
      "lat": 51.508665,
      "lon": -0.124636
    },
    "rating": 4.1,
    "distance_km": 0.26
  },
  {
    "place_id": "PLC-000731",
    "name": "Richmond Bank 00731",
    "category": "bank",
    "latitude": 51.507943,
    "longitude": -0.123858,
    "location": {
      "lat": 51.507943,
      "lon": -0.123858
    },
    "rating": 3.0,
    "distance_km": 0.279
  },
  {
    "place_id": "PLC-005197",
    "name": "Shoreditch Pharmacy 05197",
    "category": "ph

## Task 4 - Search within a geographical bounding box

The box is supplied as `top_left` and `bottom_right`. Every returned document is
checked against the numeric latitude/longitude fields.

In [5]:
BBOX_QUERY = {
    "geo_bounding_box": {
        "location": {
            "top_left": BBOX["top_left"],
            "bottom_right": BBOX["bottom_right"],
        }
    }
}
bbox_started = time.perf_counter()
bbox_response = es.search(
    index=LOCATIONS,
    query=BBOX_QUERY,
    size=2000,
    track_total_hits=True,
)
bbox_seconds = time.perf_counter() - bbox_started
bbox_total = int(bbox_response["hits"]["total"]["value"])
bbox_hits = bbox_response["hits"]["hits"]
top_left = BBOX["top_left"]
bottom_right = BBOX["bottom_right"]
all_inside = all(
    top_left["lon"] <= hit["_source"]["longitude"] <= bottom_right["lon"]
    and bottom_right["lat"] <= hit["_source"]["latitude"] <= top_left["lat"]
    for hit in bbox_hits
)

print("bounding box:", json.dumps(BBOX, indent=2))
print("matches:", f"{bbox_total:,}")
print("returned hits:", len(bbox_hits))
print("all hits inside the box:", all_inside)
print("sample:")
print(json.dumps([hit_summary(hit) for hit in bbox_hits[:5]], indent=2))

assert bbox_total > 0
assert bbox_total == len(bbox_hits)
assert all_inside
print("bounding-box query verified.")

bounding box:

 {
  "top_left": {
    "lat": 51.54,
    "lon": -0.18
  },
  "bottom_right": {
    "lat": 51.48,
    "lon": -0.07
  }
}
matches: 425
returned hits: 425
all hits inside the box: True
sample:
[
  {
    "place_id": "PLC-000004",
    "name": "Fulham Library 00004",
    "category": "library",
    "latitude": 51.493798,
    "longitude": -0.119805,
    "location": {
      "lat": 51.493798,
      "lon": -0.119805
    },
    "rating": 3.6
  },
  {
    "place_id": "PLC-000017",
    "name": "Fulham Bank 00017",
    "category": "bank",
    "latitude": 51.488474,
    "longitude": -0.133458,
    "location": {
      "lat": 51.488474,
      "lon": -0.133458
    },
    "rating": 3.2
  },
  {
    "place_id": "PLC-000018",
    "name": "Hampstead Bank 00018",
    "category": "bank",
    "latitude": 51.497322,
    "longitude": -0.130669,
    "location": {
      "lat": 51.497322,
      "lon": -0.130669
    },
    "rating": 4.5
  },
  {
    "place_id": "PLC-000040",
    "name": "Soho Restaurant 00040",
    "

## Task 5 - Sort results by distance

A `_geo_distance` sort returns the distance in kilometres in the hit's `sort`
value. The reported distance is compared with an independent haversine result.

In [6]:
sort_started = time.perf_counter()
nearest_response = es.search(
    index=LOCATIONS,
    query={"match_all": {}},
    sort=SORT,
    size=10,
    source_includes=["place_id", "name", "category", "latitude", "longitude", "location", "rating"],
)
sort_seconds = time.perf_counter() - sort_started
nearest_hits = nearest_response["hits"]["hits"]
sort_distances = [float(hit["sort"][0]) for hit in nearest_hits]
independent_distances = [
    haversine_km(REFERENCE, hit["_source"]["location"]) for hit in nearest_hits
]

print("nearest ten places:")
print(json.dumps([hit_summary(hit) for hit in nearest_hits], indent=2))
print("sort distances (km):", sort_distances)
print("independent haversine distances (km):", [round(value, 3) for value in independent_distances])

assert len(nearest_hits) == 10
assert sort_distances == sorted(sort_distances)
assert all(
    abs(sort_value - haversine_value) < 0.02
    for sort_value, haversine_value in zip(sort_distances, independent_distances)
)
print("distance sort is ascending and matches haversine.")

nearest ten places:
[
  {
    "place_id": "PLC-007804",
    "name": "Thames Bank 07804",
    "category": "bank",
    "latitude": 51.505577,
    "longitude": -0.129103,
    "location": {
      "lat": 51.505577,
      "lon": -0.129103
    },
    "rating": 4.9,
    "distance_km": 0.222
  },
  {
    "place_id": "PLC-000562",
    "name": "Hampstead Bank 00562",
    "category": "bank",
    "latitude": 51.508665,
    "longitude": -0.124636,
    "location": {
      "lat": 51.508665,
      "lon": -0.124636
    },
    "rating": 4.1,
    "distance_km": 0.26
  },
  {
    "place_id": "PLC-000731",
    "name": "Richmond Bank 00731",
    "category": "bank",
    "latitude": 51.507943,
    "longitude": -0.123858,
    "location": {
      "lat": 51.507943,
      "lon": -0.123858
    },
    "rating": 3.0,
    "distance_km": 0.279
  },
  {
    "place_id": "PLC-005197",
    "name": "Shoreditch Pharmacy 05197",
    "category": "pharmacy",
    "latitude": 51.508443,
    "longitude": -0.12347,
    "location": 

## Task 6 - Implement geo-distance aggregations

The aggregation covers every document. The final range is open-ended so that
the bucket counts sum to the full dataset.

In [7]:
aggregation_started = time.perf_counter()
aggregation_response = es.search(
    index=LOCATIONS,
    size=0,
    track_total_hits=True,
    aggs={
        "distance_rings": {
            "geo_distance": {
                "field": "location",
                "origin": REFERENCE,
                "unit": "km",
                "ranges": DISTANCE_RANGES,
            }
        }
    },
)
aggregation_seconds = time.perf_counter() - aggregation_started
bucket_rows = [
    {
        "from": bucket.get("from"),
        "to": bucket.get("to"),
        "count": int(bucket["doc_count"]),
    }
    for bucket in aggregation_response["aggregations"]["distance_rings"]["buckets"]
]
bucket_sum = sum(row["count"] for row in bucket_rows)
within_5km = sum(
    row["count"]
    for row in bucket_rows
    if (row["from"] is None or row["from"] < RADIUS_KM)
    and (row["to"] is None or row["to"] <= RADIUS_KM)
)

print("aggregation buckets:")
print(json.dumps(bucket_rows, indent=2))
print("bucket sum:", f"{bucket_sum:,}")
print("within 5 km from aggregation:", f"{within_5km:,}")
print("within 5 km from query:", f"{radius_total:,}")

assert bucket_sum == TOTAL
assert within_5km == radius_total
print("geo-distance aggregation verified.")

aggregation buckets:
[
  {
    "from": 0.0,
    "to": 1.0,
    "count": 21
  },
  {
    "from": 1.0,
    "to": 2.0,
    "count": 83
  },
  {
    "from": 2.0,
    "to": 5.0,
    "count": 556
  },
  {
    "from": 5.0,
    "to": 10.0,
    "count": 1794
  },
  {
    "from": 10.0,
    "to": 20.0,
    "count": 7546
  },
  {
    "from": 20.0,
    "to": null,
    "count": 0
  }
]
bucket sum: 10,000
within 5 km from aggregation: 660
within 5 km from query: 660
geo-distance aggregation verified.


## Task 7 - Explore `geo_shape` and polygon queries

A second index stores five district polygons as `geo_shape`. GeoJSON uses
`[longitude, latitude]` order. The rings below are closed and counter-clockwise.

Three queries are demonstrated:

1. `geo_shape` with a point: which district contains the reference point?
2. `geo_shape` with a polygon: which districts intersect an inner east-central
   envelope?
3. `geo_polygon` against the point index: which places fall inside the Central
   rectangle?

In [8]:
delete_index(DISTRICTS)
es.indices.create(index=DISTRICTS, **DISTRICT_MAPPING)
district_documents = list(build_districts())
district_load = bulk_load(DISTRICTS, district_documents, "district_id", chunk_size=10)
es.indices.put_settings(index=DISTRICTS, settings={"refresh_interval": "1s"})
es.indices.refresh(index=DISTRICTS)
districts_count = int(es.count(index=DISTRICTS, query={"match_all": {}})["count"])
districts_mapping = plain(es.indices.get_mapping(index=DISTRICTS))[DISTRICTS]["mappings"]
boundary_type = districts_mapping["properties"]["boundary"]["type"]

print("districts index:", DISTRICTS)
print("loaded:", district_load)
print("count:", districts_count)
print("boundary mapping:", json.dumps(districts_mapping["properties"]["boundary"], indent=2))

assert district_load["errors"] == 0
assert districts_count == 5
assert boundary_type == "geo_shape"
print("geo_shape mapping and district load verified.")

districts index: p12-districts
loaded: {'documents': 5, 'errors': 0, 'seconds': 0.19, 'docs_per_second': 26.3}
count: 5
boundary mapping: {
  "type": "geo_shape"
}
geo_shape mapping and district load verified.


In [9]:
center_shape = {
    "type": "point",
    "coordinates": [CENTER["lon"], CENTER["lat"]],
}
point_shape_response = es.search(
    index=DISTRICTS,
    query={
        "geo_shape": {
            "boundary": {
                "shape": center_shape,
                "relation": "intersects",
            }
        }
    },
    size=10,
    track_total_hits=True,
)
point_shape_names = [hit["_source"]["name"] for hit in point_shape_response["hits"]["hits"]]
point_shape_total = int(point_shape_response["hits"]["total"]["value"])

inner_polygon = {
    "type": "polygon",
    "coordinates": [
        [
            [-0.145, 51.495],
            [-0.005, 51.495],
            [-0.005, 51.525],
            [-0.145, 51.525],
            [-0.145, 51.495],
        ]
    ],
}
polygon_shape_response = es.search(
    index=DISTRICTS,
    query={
        "geo_shape": {
            "boundary": {
                "shape": inner_polygon,
                "relation": "intersects",
            }
        }
    },
    size=10,
    track_total_hits=True,
)
polygon_shape_names = sorted(
    hit["_source"]["name"] for hit in polygon_shape_response["hits"]["hits"]
)
polygon_shape_total = int(polygon_shape_response["hits"]["total"]["value"])

print("center GeoJSON:", json.dumps(center_shape))
print("districts containing the center:", point_shape_names)
print("polygon GeoJSON:", json.dumps(inner_polygon))
print("districts intersecting the polygon:", polygon_shape_names)

assert point_shape_total == 1
assert point_shape_names == ["Central"]
assert polygon_shape_total == 2
assert polygon_shape_names == ["Central", "East"]
print("geo_shape point and polygon queries verified.")

center GeoJSON: {"type": "point", "coordinates": [-0.1278, 51.5074]}
districts containing the center: ['Central']
polygon GeoJSON: {"type": "polygon", "coordinates": [[[-0.145, 51.495], [-0.005, 51.495], [-0.005, 51.525], [-0.145, 51.525], [-0.145, 51.495]]]}
districts intersecting the polygon: ['Central', 'East']
geo_shape point and polygon queries verified.


In [10]:
central_district = next(
    district for district in district_documents if district["district_id"] == "district-central"
)
central_ring = central_district["boundary"]["coordinates"][0]
central_points = [
    {"lat": point[1], "lon": point[0]} for point in central_ring[:-1]
]
GEO_POLYGON_QUERY = {
    "geo_polygon": {
        "location": {
            "points": central_points,
        }
    }
}
geo_polygon_started = time.perf_counter()
geo_polygon_response = es.search(
    index=LOCATIONS,
    query=GEO_POLYGON_QUERY,
    size=2000,
    track_total_hits=True,
)
geo_polygon_seconds = time.perf_counter() - geo_polygon_started
geo_polygon_total = int(geo_polygon_response["hits"]["total"]["value"])
geo_polygon_hits = geo_polygon_response["hits"]["hits"]

central_lon_min, central_lon_max = -0.15, -0.10
central_lat_min, central_lat_max = 51.49, 51.53
independent_polygon_count = sum(
    1
    for place in build_places(count=TOTAL)
    if central_lon_min <= place["longitude"] <= central_lon_max
    and central_lat_min <= place["latitude"] <= central_lat_max
)

print("geo_polygon points:", json.dumps(central_points))
print("matches from Elasticsearch:", f"{geo_polygon_total:,}")
print("matches from independent Python check:", f"{independent_polygon_count:,}")
print("sample:")
print(json.dumps([hit_summary(hit) for hit in geo_polygon_hits[:5]], indent=2))

assert geo_polygon_total == independent_polygon_count
assert geo_polygon_total > 0
assert len(geo_polygon_hits) == geo_polygon_total
print("geo_polygon query matches the independent point-in-polygon count.")

geo_polygon points: [{"lat": 51.49, "lon": -0.15}, {"lat": 51.49, "lon": -0.1}, {"lat": 51.53, "lon": -0.1}, {"lat": 51.53, "lon": -0.15}]
matches from Elasticsearch: 126
matches from independent Python check: 126
sample:
[
  {
    "place_id": "PLC-000004",
    "name": "Fulham Library 00004",
    "category": "library",
    "latitude": 51.493798,
    "longitude": -0.119805,
    "location": {
      "lat": 51.493798,
      "lon": -0.119805
    },
    "rating": 3.6
  },
  {
    "place_id": "PLC-000018",
    "name": "Hampstead Bank 00018",
    "category": "bank",
    "latitude": 51.497322,
    "longitude": -0.130669,
    "location": {
      "lat": 51.497322,
      "lon": -0.130669
    },
    "rating": 4.5
  },
  {
    "place_id": "PLC-000040",
    "name": "Soho Restaurant 00040",
    "category": "restaurant",
    "latitude": 51.495438,
    "longitude": -0.14797,
    "location": {
      "lat": 51.495438,
      "lon": -0.14797
    },
    "rating": 4.1
  },
  {
    "place_id": "PLC-000400",
  

C:\Users\Dell\AppData\Local\Temp\ipykernel_11700\3493760765.py:16: ElasticsearchWarning: Deprecated field [geo_polygon] used, replaced by [[geo_shape] query where polygons are defined in geojson or wkt]
  geo_polygon_response = es.search(


## Query timing summary

Each query is repeated ten times after the result assertions. These timings are
local measurements and are not a production benchmark.

In [11]:
def measure_query(index, iterations=10, **search_kwargs):
    timings = []
    last_response = None
    for _ in range(iterations):
        started = time.perf_counter()
        last_response = es.search(index=index, **search_kwargs)
        timings.append((time.perf_counter() - started) * 1000)
    ordered = sorted(timings)
    p95_index = max(0, math.ceil(0.95 * len(ordered)) - 1)
    return {
        "iterations": iterations,
        "p50_ms": round(statistics.median(timings), 3),
        "p95_ms": round(ordered[p95_index], 3),
        "total_hits": int(last_response["hits"]["total"]["value"]),
    }


benchmark = {
    "dataset_documents": TOTAL,
    "iterations": 10,
    "queries": {
        "radius_5km": measure_query(
            LOCATIONS,
            query=RADIUS_QUERY,
            size=0,
            track_total_hits=True,
        ),
        "bounding_box": measure_query(
            LOCATIONS,
            query=BBOX_QUERY,
            size=0,
            track_total_hits=True,
        ),
        "distance_sort": measure_query(
            LOCATIONS,
            query={"match_all": {}},
            sort=SORT,
            size=10,
        ),
        "geo_distance_aggregation": measure_query(
            LOCATIONS,
            size=0,
            track_total_hits=True,
            aggs={
                "distance_rings": {
                    "geo_distance": {
                        "field": "location",
                        "origin": REFERENCE,
                        "unit": "km",
                        "ranges": DISTANCE_RANGES,
                    }
                }
            },
        ),
        "geo_shape_point": measure_query(
            DISTRICTS,
            query={
                "geo_shape": {
                    "boundary": {"shape": center_shape, "relation": "intersects"}
                }
            },
            size=10,
            track_total_hits=True,
        ),
        "geo_shape_polygon": measure_query(
            DISTRICTS,
            query={
                "geo_shape": {
                    "boundary": {"shape": inner_polygon, "relation": "intersects"}
                }
            },
            size=10,
            track_total_hits=True,
        ),
        "geo_polygon": measure_query(
            LOCATIONS,
            query=GEO_POLYGON_QUERY,
            size=0,
            track_total_hits=True,
        ),
    },
}

print(json.dumps(benchmark, indent=2))
assert all(
    query["total_hits"] >= 0 for query in benchmark["queries"].values()
)

C:\Users\Dell\AppData\Local\Temp\ipykernel_11700\1716791512.py:6: ElasticsearchWarning: Deprecated field [geo_polygon] used, replaced by [[geo_shape] query where polygons are defined in geojson or wkt]
  last_response = es.search(index=index, **search_kwargs)


{
  "dataset_documents": 10000,
  "iterations": 10,
  "queries": {
    "radius_5km": {
      "iterations": 10,
      "p50_ms": 52.362,
      "p95_ms": 80.181,
      "total_hits": 660
    },
    "bounding_box": {
      "iterations": 10,
      "p50_ms": 50.144,
      "p95_ms": 56.117,
      "total_hits": 425
    },
    "distance_sort": {
      "iterations": 10,
      "p50_ms": 56.482,
      "p95_ms": 74.614,
      "total_hits": 10000
    },
    "geo_distance_aggregation": {
      "iterations": 10,
      "p50_ms": 50.327,
      "p95_ms": 84.328,
      "total_hits": 10000
    },
    "geo_shape_point": {
      "iterations": 10,
      "p50_ms": 47.755,
      "p95_ms": 76.414,
      "total_hits": 1
    },
    "geo_shape_polygon": {
      "iterations": 10,
      "p50_ms": 52.822,
      "p95_ms": 242.141,
      "total_hits": 2
    },
    "geo_polygon": {
      "iterations": 10,
      "p50_ms": 64.631,
      "p95_ms": 76.849,
      "total_hits": 126
    }
  }
}


## Write artifacts

Both JSON artifacts are written under `artifacts/`, then the full report is
checked for the expected document counts.

In [12]:
district_metrics = index_metrics(DISTRICTS)

geo_report = {
    "environment": {
        "elasticsearch_version": info["version"]["number"],
        "cluster_name": info["cluster_name"],
        "node_name": info["name"],
    },
    "dataset": {
        "locations_index": LOCATIONS,
        "districts_index": DISTRICTS,
        "documents": indexed_count,
        "districts": districts_count,
        "center": CENTER,
        "maximum_generated_radius_km": MAX_RADIUS_KM,
        "place_fields": PLACE_FIELDS,
        "district_fields": DISTRICT_FIELDS,
        "load": load_info,
        "district_load": district_load,
        "location_metrics": location_metrics,
        "district_metrics": district_metrics,
    },
    "mapping": {
        "location_type": location_type,
        "boundary_type": boundary_type,
        "location_mapping": locations_mapping,
        "district_properties": districts_mapping["properties"],
    },
    "radius": {
        "center": REFERENCE,
        "distance_km": RADIUS_KM,
        "query": RADIUS_QUERY,
        "matches": radius_total,
        "returned": len(radius_hits),
        "max_distance_km": round(max(radius_distances), 4),
        "elapsed_seconds": round(radius_seconds, 4),
        "sample": [hit_summary(hit) for hit in radius_hits[:10]],
    },
    "bounding_box": {
        "box": BBOX,
        "query": BBOX_QUERY,
        "matches": bbox_total,
        "returned": len(bbox_hits),
        "all_inside": all_inside,
        "elapsed_seconds": round(bbox_seconds, 4),
        "sample": [hit_summary(hit) for hit in bbox_hits[:10]],
    },
    "distance_sort": {
        "origin": REFERENCE,
        "sort": SORT,
        "returned": len(nearest_hits),
        "elapsed_seconds": round(sort_seconds, 4),
        "places": [hit_summary(hit) for hit in nearest_hits],
        "sort_distances_km": sort_distances,
        "haversine_distances_km": [round(value, 3) for value in independent_distances],
    },
    "aggregation": {
        "origin": REFERENCE,
        "unit": "km",
        "ranges": DISTANCE_RANGES,
        "buckets": bucket_rows,
        "bucket_sum": bucket_sum,
        "within_5km": within_5km,
        "elapsed_seconds": round(aggregation_seconds, 4),
    },
    "shapes": {
        "districts": districts_count,
        "point_shape": center_shape,
        "point_shape_matches": point_shape_names,
        "point_shape_total": point_shape_total,
        "polygon_shape": inner_polygon,
        "polygon_shape_matches": polygon_shape_names,
        "polygon_shape_total": polygon_shape_total,
        "geo_polygon": {
            "query": GEO_POLYGON_QUERY,
            "matches": geo_polygon_total,
            "independent_count": independent_polygon_count,
            "elapsed_seconds": round(geo_polygon_seconds, 4),
            "sample": [hit_summary(hit) for hit in geo_polygon_hits[:10]],
        },
    },
}

(ARTIFACTS / "geo_report.json").write_text(
    json.dumps(plain(geo_report), indent=2), encoding="utf-8"
)
(ARTIFACTS / "benchmark.json").write_text(
    json.dumps(benchmark, indent=2), encoding="utf-8"
)

print("wrote:", ARTIFACTS / "geo_report.json")
print("wrote:", ARTIFACTS / "benchmark.json")
print("report locations:", geo_report["dataset"]["documents"])
print("report districts:", geo_report["dataset"]["districts"])
print("radius matches:", geo_report["radius"]["matches"])
print("bounding-box matches:", geo_report["bounding_box"]["matches"])
print("aggregation sum:", geo_report["aggregation"]["bucket_sum"])
print("geo_polygon matches:", geo_report["shapes"]["geo_polygon"]["matches"])

assert geo_report["dataset"]["documents"] == TOTAL
assert geo_report["dataset"]["districts"] == 5
assert geo_report["radius"]["matches"] == geo_report["aggregation"]["within_5km"]
assert geo_report["aggregation"]["bucket_sum"] == TOTAL
assert geo_report["shapes"]["geo_polygon"]["matches"] == independent_polygon_count
assert geo_report["mapping"]["location_type"] == "geo_point"
assert geo_report["mapping"]["boundary_type"] == "geo_shape"
print("all artifact assertions passed.")

wrote: C:\D-Drive\Elastic Search Practicals\Practical_12\artifacts\geo_report.json
wrote: C:\D-Drive\Elastic Search Practicals\Practical_12\artifacts\benchmark.json
report locations: 10000
report districts: 5
radius matches: 660
bounding-box matches: 425
aggregation sum: 10000
geo_polygon matches: 126
all artifact assertions passed.
